In [2]:
!pip install numpy pandas

  Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 11.7 MB/s  0:00:00 eta 0:00:01
Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl (9.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [8]:
import numpy as np
import pandas as pd

In [10]:
def heat_fdm(initial, left_bc, right_bc, L=1, T=1, c=1):
    """
    Finite difference solver for the heat equation.

    initial : list
        values of u(x_i,0)

    left_bc : list
        boundary values at x=0 for each time step

    right_bc : list
        boundary values at x=L for each time step
    
    L = length of rod

    T = time to solve for

    c = diffusion constant
    """

    n = len(initial) - 1
    m = len(left_bc) - 1

    h = L / n
    k = T / m

    s = c**2 * k / h**2

    # grid
    x = np.linspace(0, L, n+1)
    t = np.linspace(0, T, m+1)

    # solution array
    u = np.zeros((m+1, n+1))

    # initial condition
    u[0, :] = initial

    # boundary conditions
    for j in range(m+1):
        u[j, 0] = left_bc[j]
        u[j, -1] = right_bc[j]

    # finite difference iteration
    # note: runtime O(nm)
    for j in range(m):
        for i in range(1, n):
            u[j+1, i] = u[j, i] + s * (u[j, i+1] - 2*u[j, i] + u[j, i-1])

    # return grid table
    grid = pd.DataFrame(u, columns=[f"x{i:.2f}" for i in x])
    grid.insert(0, "t", t)

    return grid

In [15]:
def calculate_stability(h, k, c):
    """
    Calculate the stability of the finite difference method.

    h = step size in space

    k = step size in time

    c = diffusion constant
    """
    return c**2 * k / h**2

Dirichlet Boundary Conditions: random initial temperatures

In [16]:
import random

initial = [random.randint(-100, 100) for _ in range(10)]
left_bc = [0, 0, 0, 0, 0, 0, 0]
right_bc = [0, 0, 0, 0, 0, 0, 0]

print("Inital Condition:", initial)
ans = heat_fdm(initial, left_bc, right_bc)

print("Solution:")
print(ans)

stability = calculate_stability(len(initial), len(right_bc), 1)
if stability > 0.5:
    print("Stability:", stability)
    print("The method is unstable.")
else:
    print("Stability:", stability)
    print("The method is stable.")


Inital Condition: [73, -16, -13, -90, -34, 20, -66, -77, -47, 64]
Solution:
          t  x0.00         x0.11         x0.22         x0.33         x0.44  \
0  0.000000    0.0 -1.600000e+01 -1.300000e+01 -9.000000e+01 -3.400000e+01   
1  0.166667    0.0  2.405000e+02 -1.093000e+03  1.705500e+03 -6.100000e+01   
2  0.333333    0.0 -2.100850e+04  5.468900e+04 -5.992200e+04 -6.347500e+02   
3  0.500000    0.0  1.284522e+06 -2.514476e+06  2.287704e+06  2.530888e+04   
4  0.666667    0.0 -6.734301e+07  1.136014e+08 -9.308407e+07  9.352106e+05   
5  0.833333    0.0  3.284538e+09 -5.119403e+09  3.966430e+09 -1.529727e+08   
6  1.000000    0.0 -1.545099e+11  2.309925e+11 -1.743043e+11  1.205329e+10   

          x0.56         x0.67         x0.78         x0.89  x1.00  
0  2.000000e+01 -6.600000e+01 -7.700000e+01 -4.700000e+01    0.0  
1 -1.870000e+03  9.465000e+02  4.765000e+02  1.825000e+02    0.0  
2  6.057425e+04 -4.342125e+04  2.852500e+03  1.687750e+03    0.0  
3 -2.169686e+06  1.985214e+06 -